In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("1. Loading and balancing data...")
# Load your new masterpiece
df = pd.read_csv('dataset_final.csv', sep=';')

# Fix the stray header row from stitching
df = df[df['Label'] != 'Label']

# Balance perfectly (brings everyone down to the lowest count, e.g., 779)
min_count = df['Label'].value_counts().min()
df_balanced = df.groupby('Label').sample(n=min_count, random_state=42)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

textos = df_balanced['Text'].values
labels = df_balanced['Label'].values

print(f"Dataset ready! Total perfectly balanced rows: {len(df_balanced)}")

print("\n2. Running TF-IDF (Character Level)...")
# The Secret Weapon: Look at punctuation and character patterns, not vocabulary!
vectorizer = TfidfVectorizer(
    analyzer='char',         
    ngram_range=(3, 6),      # Expanded slightly to catch longer suffixes (6 chars)
    max_features=12000,      # Pushed to 12k since we are using max_df/min_df
    lowercase=False,         # CRITICAL: Keep capital letters!
    sublinear_tf=True,       # CRITICAL: Smooths out frequency spikes
    min_df=5,                # Ignore extremely rare typos
    max_df=0.85              # Ignore features that appear in 85%+ of texts
)
X = vectorizer.fit_transform(textos).toarray()

# Save the vectorizer for your submission
import os
os.makedirs('Subm1', exist_ok=True)
with open('Subm1/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("\n3. Encoding Labels (Using Professor's Custom Map)...")

# Force the exact mapping you want
PROFESSOR_MAP = {
    'Human': 0,
    'OpenAI': 1,
    'Google': 2,
    'Meta': 3,
    'Anthropic': 4
}

# Apply the map to your labels list
y = np.array([PROFESSOR_MAP[label] for label in labels])

print("Mapped successfully using custom integer dictionary!")

print("\n4. Splitting and Saving Data...")
# Split into 80% Train, 10% Validation, 10% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Save the arrays
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

print(f"\nSuccess! X_train shape: {X_train.shape}")

1. Loading and balancing data...
Dataset ready! Total perfectly balanced rows: 4355

2. Running TF-IDF (Character Level)...

3. Encoding Labels (Using Professor's Custom Map)...
Mapped successfully using custom integer dictionary!

4. Splitting and Saving Data...

Success! X_train shape: (3484, 12000)
